# LeetCode 1637 — Widest Vertical Area Between Two Points Containing No Points bằng Python

Link đề: https://leetcode.com/problems/widest-vertical-area-between-two-points-containing-no-points/

Độ khó: **Easy** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxWidthOfVerticalArea(self, points: list[list[int]]) -> int:

```

Mục tiêu học:

1. Đọc hiểu đề hình học và **nhận ra tọa độ y không liên quan**.
2. Biết tách cột từ list các cặp bằng list comprehension.
3. Từ brute force `O(n²)` (thử mọi cặp) đi tới **sort + so sánh hai phần tử kề nhau** `O(n log n)`.
4. Làm quen `zip(a, a[1:])` để duyệt các cặp kề nhau.
5. Biết thêm kỹ thuật **bucket / pigeonhole** cho bài "max gap" trong `O(n)` (giống LeetCode 164).
6. Tự viết lại lời giải và giải các biến thể "khoảng trống lớn nhất".

## 1. Bài toán yêu cầu gì?

Cho `n` điểm trên mặt phẳng, `points[i] = [xi, yi]`.

Một **vertical area** (vùng dọc) là một dải có bề rộng cố định, kéo dài vô hạn theo trục y — tức là dải `a < x < b` với mọi y.

Ta cần tìm dải dọc **rộng nhất** nằm giữa hai điểm sao cho **không có điểm nào nằm bên trong** dải. Điểm nằm **đúng trên mép** dải không tính là bên trong.

Trả về **bề rộng** `b - a` lớn nhất.

| points | Kết quả | Giải thích ngắn |
|---|---:|---|
| `[[8,7],[9,9],[7,4],[9,7]]` | `1` | các x: 7, 8, 9, 9 → khe lớn nhất 7→8 hoặc 8→9 đều rộng 1 |
| `[[3,1],[9,0],[1,0],[1,4],[5,3],[8,8]]` | `3` | các x sau khi sắp: 1,1,3,5,8,9 → khe 5→8 rộng 3 |

Constraints:

- `2 <= n <= 10^5`
- `points[i].length == 2`
- `0 <= xi, yi <= 10^9`

### Hình dung bằng tay

Dải dọc chỉ bị "chặn" bởi tọa độ **x**. Một điểm `(x, y)` nằm trong dải `a < x < b` hay không chỉ phụ thuộc x, **không phụ thuộc y** (vì dải cao vô hạn).

Vậy bài toán thực chất là: chiếu tất cả điểm xuống trục x, rồi tìm **khoảng trống lớn nhất giữa hai x liên tiếp**.

Ví dụ 2, chiếu xuống trục x:

```text
x:  0  1  2  3  4  5  6  7  8  9
       ●     ●     ●        ●  ●
       ●
khe:   1→3 (2)  3→5 (2)  5→8 (3)  8→9 (1)
```

Khe lớn nhất là `5 → 8`, rộng `3`.

In [1]:
points = [[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]
xs = [p[0] for p in points]
print("Các tọa độ x:", xs)
print("Sau khi sắp :", sorted(xs))

Các tọa độ x: [3, 9, 1, 1, 5, 8]
Sau khi sắp : [1, 1, 3, 5, 8, 9]


## 2. Nền tảng Python cần biết

### 2.1. Truy cập phần tử của list lồng nhau

`points` là list các list 2 phần tử. `points[i]` là một điểm, `points[i][0]` là x, `points[i][1]` là y.

Có thể **gỡ** (unpack) hai phần tử cùng lúc: `x, y = point`.

In [2]:
points = [[8, 7], [9, 9], [7, 4], [9, 7]]
print("points[2]    =", points[2])
print("points[2][0] =", points[2][0], "(x)")
print("points[2][1] =", points[2][1], "(y)")

for x, y in points:
    print(f"x = {x}, y = {y}")

points[2]    = [7, 4]
points[2][0] = 7 (x)
points[2][1] = 4 (y)
x = 8, y = 7
x = 9, y = 9
x = 7, y = 4
x = 9, y = 7


### 2.2. List comprehension để tách cột x

`[p[0] for p in points]` nghĩa là: "với mỗi `p` trong `points`, lấy `p[0]`, gom thành list mới".
Viết tương đương bằng vòng for:

In [3]:
points = [[8, 7], [9, 9], [7, 4], [9, 7]]

xs_loop = []
for p in points:
    xs_loop.append(p[0])

xs_comp = [p[0] for p in points]
xs_unpack = [x for x, _ in points]   # dấu _ = "biến bỏ qua"

print(xs_loop, xs_comp, xs_unpack)

[8, 9, 7, 9] [8, 9, 7, 9] [8, 9, 7, 9]


### 2.3. Sắp xếp: `sorted()` và `.sort()`

- `sorted(xs)` tạo list mới tăng dần.
- `xs.sort()` sắp tại chỗ và trả về `None`.

Sắp xếp `n` phần tử tốn `O(n log n)`.

In [4]:
xs = [8, 9, 7, 9]
print(sorted(xs), "| xs vẫn là", xs)
xs.sort()
print("sau xs.sort():", xs)

[7, 8, 9, 9] | xs vẫn là [8, 9, 7, 9]
sau xs.sort(): [7, 8, 9, 9]


### 2.4. Duyệt cặp kề nhau: theo chỉ số và theo `zip`

Sau khi sắp, ta cần các cặp `(xs[i-1], xs[i])`.

- Cách chỉ số: `for i in range(1, len(xs))` — bắt đầu từ 1 để `xs[i-1]` hợp lệ.
- Cách `zip`: `zip(xs, xs[1:])` ghép phần tử thứ i của `xs` với phần tử thứ i của `xs[1:]` (chính là phần tử kế tiếp).

`xs[1:]` là **slicing**: lấy từ chỉ số 1 đến hết.

In [5]:
xs = [1, 1, 3, 5, 8, 9]
print("xs[1:] =", xs[1:])

print("Theo chỉ số:")
for i in range(1, len(xs)):
    print(f"  i={i}: ({xs[i-1]}, {xs[i]}) khe = {xs[i] - xs[i-1]}")

print("Theo zip:", list(zip(xs, xs[1:])))

xs[1:] = [1, 3, 5, 8, 9]
Theo chỉ số:
  i=1: (1, 1) khe = 0
  i=2: (1, 3) khe = 2
  i=3: (3, 5) khe = 2
  i=4: (5, 8) khe = 3
  i=5: (8, 9) khe = 1
Theo zip: [(1, 1), (1, 3), (3, 5), (5, 8), (8, 9)]


### 2.5. `max()` với generator

`max(b - a for a, b in zip(xs, xs[1:]))` tính khe của từng cặp rồi lấy lớn nhất, không cần tạo list trung gian.

In [6]:
xs = [1, 1, 3, 5, 8, 9]
gaps = [b - a for a, b in zip(xs, xs[1:])]
print("Các khe:", gaps)
print("Khe lớn nhất:", max(gaps))
print("Viết gọn:", max(b - a for a, b in zip(xs, xs[1:])))

Các khe: [0, 2, 2, 3, 1]
Khe lớn nhất: 3
Viết gọn: 3


## 3. Cách trực giác — Brute Force thử mọi cặp

Định nghĩa đề nói "giữa hai điểm". Cách nghĩ thẳng:

1. Thử mọi cặp điểm `(i, j)` với `x_i < x_j`.
2. Kiểm tra có điểm `k` nào có `x_i < x_k < x_j` không.
3. Nếu không có điểm nào chen giữa, cập nhật đáp án `x_j - x_i`.

Cách này bám sát định nghĩa nhưng tốn `O(n³)` (hai vòng chọn cặp + một vòng kiểm tra).

In [7]:
def widest_bruteforce(points: list[list[int]]) -> int:
    n = len(points)
    best = 0
    for i in range(n):
        for j in range(n):
            a, b = points[i][0], points[j][0]
            if a >= b:
                continue
            empty = True
            for k in range(n):
                if a < points[k][0] < b:
                    empty = False
                    break
            if empty:
                best = max(best, b - a)
    return best

print(widest_bruteforce([[8, 7], [9, 9], [7, 4], [9, 7]]))
print(widest_bruteforce([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

1
3


Chú ý cú pháp `a < points[k][0] < b`: Python cho phép **so sánh nối chuỗi**, tương đương `a < points[k][0] and points[k][0] < b`. Dấu `<` nghiêm ngặt đúng với ý "điểm trên mép không tính".

In [8]:
a, b = 5, 8
for x in [5, 6, 8, 9]:
    print(x, "nằm trong (5, 8)?", a < x < b)

5 nằm trong (5, 8)? False
6 nằm trong (5, 8)? True
8 nằm trong (5, 8)? False
9 nằm trong (5, 8)? False


### Nút thắt

- Với `n = 10^5`, `n³ = 10^15` phép tính — không thể chạy kịp.
- Ta đang **thử lại** rất nhiều cặp vô ích: cặp `(1, 9)` chắc chắn có điểm chen giữa.

Câu hỏi then chốt: **cặp nào có thể "không có điểm chen giữa"?**

Nếu sắp các x tăng dần, giữa hai x **liền kề** không thể có x nào khác. Còn giữa hai x không liền kề thì luôn có ít nhất một x chen giữa (hoặc khe đó bằng tổng các khe nhỏ hơn bên trong — không thể là khe "trống").

In [9]:
def count_pairs_checked(n):
    return n * n * n

for n in [10, 100, 1000, 10**5]:
    print(f"n = {n:>6}: brute force ~ {count_pairs_checked(n):.1e} bước")

n =     10: brute force ~ 1.0e+03 bước
n =    100: brute force ~ 1.0e+06 bước
n =   1000: brute force ~ 1.0e+09 bước
n = 100000: brute force ~ 1.0e+15 bước


## 4. Insight — chỉ cần các x liền kề sau khi sắp

Hai quan sát:

1. **Bỏ y**: dải cao vô hạn nên y không bao giờ làm điểm "thoát" khỏi dải.
2. **Sort x**: sau khi sắp `xs` tăng dần, mọi khe trống là một khe `xs[i] - xs[i-1]`. Khe trống lớn nhất là `max(xs[i] - xs[i-1])`.

x trùng nhau (ví dụ `1, 1` hoặc `9, 9`) cho khe `0`, không ảnh hưởng `max` vì đáp án luôn `>= 0`.

Tính tay ví dụ 1: `xs = [8, 9, 7, 9]` → sắp `[7, 8, 9, 9]` → khe `1, 1, 0` → max `1`.

In [10]:
xs = sorted([8, 9, 7, 9])
print("xs đã sắp:", xs)
for i in range(1, len(xs)):
    print(f"khe giữa {xs[i-1]} và {xs[i]} = {xs[i] - xs[i-1]}")

xs đã sắp: [7, 8, 9, 9]
khe giữa 7 và 8 = 1
khe giữa 8 và 9 = 1
khe giữa 9 và 9 = 0


## 5. Lời giải chính — sort x rồi tìm khe kề lớn nhất

Giảng từng đoạn:

**Đoạn 1 — lấy x và sắp:**

```python
xs = sorted(x for x, _ in points)
```

**Đoạn 2 — duyệt từng cặp kề, giữ khe lớn nhất:**

```python
best = 0
for i in range(1, len(xs)):
    best = max(best, xs[i] - xs[i - 1])
```

**Đoạn 3 — trả về `best`.**

In [11]:
def widest_sort(points: list[list[int]]) -> int:
    xs = sorted(x for x, _ in points)
    best = 0
    for i in range(1, len(xs)):
        best = max(best, xs[i] - xs[i - 1])
    return best

print(widest_sort([[8, 7], [9, 9], [7, 4], [9, 7]]))
print(widest_sort([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

1
3


## 6. TRACE từng bước

Bảng trace cho ví dụ 2 (`xs` đã sắp `= [1, 1, 3, 5, 8, 9]`):

| i | xs[i-1] | xs[i] | khe | best trước | best sau |
|---:|---:|---:|---:|---:|---:|
| 1 | 1 | 1 | 0 | 0 | 0 |
| 2 | 1 | 3 | 2 | 0 | 2 |
| 3 | 3 | 5 | 2 | 2 | 2 |
| 4 | 5 | 8 | 3 | 2 | **3** |
| 5 | 8 | 9 | 1 | 3 | 3 |

Cột "best trước" cho thấy ta so sánh khe mới với kỷ lục cũ rồi mới cập nhật.

In [12]:
def trace_widest(points: list[list[int]]) -> int:
    xs = sorted(x for x, _ in points)
    print("xs đã sắp:", xs)
    best = 0
    for i in range(1, len(xs)):
        gap = xs[i] - xs[i - 1]
        old = best
        best = max(best, gap)
        note = "cập nhật" if best > old else "giữ nguyên"
        print(f"i={i}: ({xs[i-1]:>2}, {xs[i]:>2}) khe={gap} | best {old} -> {best} ({note})")
    print("Đáp án:", best)
    return best

trace_widest([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]])

xs đã sắp: [1, 1, 3, 5, 8, 9]
i=1: ( 1,  1) khe=0 | best 0 -> 0 (giữ nguyên)
i=2: ( 1,  3) khe=2 | best 0 -> 2 (cập nhật)
i=3: ( 3,  5) khe=2 | best 2 -> 2 (giữ nguyên)
i=4: ( 5,  8) khe=3 | best 2 -> 3 (cập nhật)
i=5: ( 8,  9) khe=1 | best 3 -> 3 (giữ nguyên)
Đáp án: 3


3

In [13]:
trace_widest([[8, 7], [9, 9], [7, 4], [9, 7]])

xs đã sắp: [7, 8, 9, 9]
i=1: ( 7,  8) khe=1 | best 0 -> 1 (cập nhật)
i=2: ( 8,  9) khe=1 | best 1 -> 1 (giữ nguyên)
i=3: ( 9,  9) khe=0 | best 1 -> 1 (giữ nguyên)
Đáp án: 1


1

## 7. Code nộp LeetCode

In [14]:
class Solution:
    def maxWidthOfVerticalArea(self, points: list[list[int]]) -> int:
        xs = sorted(x for x, _ in points)
        return max(b - a for a, b in zip(xs, xs[1:]))

sol = Solution()
assert sol.maxWidthOfVerticalArea([[8, 7], [9, 9], [7, 4], [9, 7]]) == 1
assert sol.maxWidthOfVerticalArea([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]) == 3
print("2 ví dụ của đề PASS")

2 ví dụ của đề PASS


Vì `n >= 2`, `zip(xs, xs[1:])` luôn có ít nhất một cặp, nên `max(...)` không bị lỗi "max() arg is an empty sequence".

## 8. Các lời giải khác

### 8.1. Bỏ trùng bằng `set` trước khi sort

`set` là tập hợp không chứa phần tử trùng. `sorted(set(xs))` vừa bỏ trùng vừa sắp. Khe `0` biến mất, kết quả không đổi. Có lợi khi nhiều điểm trùng x.

In [15]:
xs = [1, 9, 1, 3, 5, 8]
print("set(xs)         =", set(xs))
print("sorted(set(xs)) =", sorted(set(xs)))

def widest_set(points: list[list[int]]) -> int:
    xs = sorted({x for x, _ in points})   # set comprehension
    if len(xs) < 2:
        return 0                           # mọi điểm cùng x
    return max(b - a for a, b in zip(xs, xs[1:]))

print(widest_set([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))
print(widest_set([[5, 1], [5, 9]]))

set(xs)         = {1, 3, 5, 8, 9}
sorted(set(xs)) = [1, 3, 5, 8, 9]
3
0


Chú ý: sau khi bỏ trùng có thể chỉ còn 1 giá trị (ví dụ mọi điểm đều có x = 5), khi đó phải trả `0` — đây là edge case mà cách 8.1 phải tự xử lý.

### 8.2. Bucket / pigeonhole — `O(n)` không cần sort

Đây là kỹ thuật của LeetCode 164 Maximum Gap. Ý tưởng:

1. Gọi `lo = min(xs)`, `hi = max(xs)`, `m` = số giá trị. Khe lớn nhất luôn `>= (hi - lo) / (m - 1)` (trung bình các khe).
2. Chia đoạn `[lo, hi]` thành các bucket bề rộng `size = max(1, (hi - lo) // (m - 1))`.
3. Khe lớn nhất **không nằm trong một bucket** (vì trong bucket khe < size ≤ trung bình), nên chỉ cần lưu **min và max của mỗi bucket**.
4. Đáp án = max của `min(bucket sau) - max(bucket trước có dữ liệu)`.

Đây là phần nâng cao; với bài Easy, sort là đủ.

In [16]:
def widest_bucket(points: list[list[int]]) -> int:
    xs = list({x for x, _ in points})
    m = len(xs)
    if m < 2:
        return 0
    lo, hi = min(xs), max(xs)
    size = max(1, (hi - lo) // (m - 1))
    count = (hi - lo) // size + 1
    bmin = [None] * count
    bmax = [None] * count
    for x in xs:
        idx = (x - lo) // size
        bmin[idx] = x if bmin[idx] is None else min(bmin[idx], x)
        bmax[idx] = x if bmax[idx] is None else max(bmax[idx], x)
    best = 0
    prev_max = lo
    for idx in range(count):
        if bmin[idx] is None:
            continue
        best = max(best, bmin[idx] - prev_max)
        prev_max = bmax[idx]
    return best

print(widest_bucket([[8, 7], [9, 9], [7, 4], [9, 7]]))
print(widest_bucket([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

1
3


Giải thích vài cú pháp mới:

- `[None] * count`: list gồm `count` giá trị `None` (nghĩa là "bucket rỗng").
- `A if điều_kiện else B`: biểu thức điều kiện một dòng.
- `x is None`: kiểm tra đúng là `None` (dùng `is`, không dùng `==`).

In [17]:
print([None] * 3)
v = None
print("rỗng" if v is None else "có dữ liệu")
v = 7
print("rỗng" if v is None else "có dữ liệu")

[None, None, None]
rỗng
có dữ liệu


### 8.3. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|---|---|---|
| Brute force 3 vòng | thử mọi cặp, kiểm tra điểm chen giữa | `O(n³)` | `O(1)` | bám định nghĩa, quá chậm |
| Sort + khe kề | sắp x, max hiệu kề | `O(n log n)` | `O(n)` | **nên dùng** |
| Set + sort | bỏ trùng trước | `O(n log n)` | `O(n)` | phải xử lý còn 1 giá trị |
| Bucket / pigeonhole | min/max mỗi bucket | `O(n)` | `O(n)` | nâng cao, giống LC 164 |

## 9. Vì sao lời giải đúng?

Gọi `xs` là các x đã sắp: `xs[0] <= xs[1] <= ... <= xs[n-1]`.

**(a) Mọi khe kề đều là vùng trống hợp lệ.** Dải `xs[i-1] < x < xs[i]` không chứa điểm nào, vì nếu có điểm với x nằm giữa thì khi sắp nó phải đứng giữa `xs[i-1]` và `xs[i]`, mâu thuẫn với "liền kề".

**(b) Mọi vùng trống đều nằm gọn trong một khe kề.** Một dải trống giữa hai điểm có mép trái `a` và mép phải `b` là x của hai điểm. Không có x nào nằm giữa `a` và `b`, nên `a`, `b` phải liền kề nhau trong `xs` (hoặc bằng nhau).

Từ (a) và (b): tập các vùng trống chính là tập các khe kề, nên đáp án là khe kề lớn nhất.

Invariant vòng lặp: sau khi xử lý tới `i`, `best = max(xs[j] - xs[j-1] với 1 <= j <= i)`.

## 10. Độ phức tạp

Gọi `n = len(points)`.

- **Sort + khe kề:** tạo `xs` `O(n)`, sort `O(n log n)`, duyệt `O(n)` → **Time `O(n log n)`**. Bộ nhớ `O(n)` cho `xs` (và `xs[1:]` nếu dùng zip; cách chỉ số tránh được bản sao này).
- **Brute force:** `O(n³)` thời gian, `O(1)` bộ nhớ.
- **Bucket:** `O(n)` thời gian và bộ nhớ (số bucket `<= n`).

Tọa độ lên tới `10^9` nhưng Python `int` không tràn, nên hiệu `b - a` luôn chính xác.

## 11. Bộ test

1. 2 ví dụ của đề.
2. Biên: đúng 2 điểm; 2 điểm cùng x → `0`.
3. Mọi điểm cùng x → `0`.
4. Tọa độ lớn `10^9`.
5. y khác nhau nhưng x giống — kiểm tra rằng y bị bỏ qua.
6. Đối chiếu brute force / sort / set / bucket trên input ngẫu nhiên.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS -> {actual}")

check("đề 1", widest_sort([[8, 7], [9, 9], [7, 4], [9, 7]]), 1)
check("đề 2", widest_sort([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]), 3)
check("2 điểm", widest_sort([[0, 0], [10, 10]]), 10)
check("2 điểm cùng x", widest_sort([[4, 0], [4, 100]]), 0)
check("mọi điểm cùng x", widest_sort([[2, 1], [2, 5], [2, 9]]), 0)
check("tọa độ 10^9", widest_sort([[0, 0], [10**9, 0], [1, 5]]), 10**9 - 1)
check("y không ảnh hưởng", widest_sort([[1, 0], [5, 10**9], [6, 3]]), 4)

đề 1                       PASS -> 1
đề 2                       PASS -> 3
2 điểm                     PASS -> 10
2 điểm cùng x              PASS -> 0
mọi điểm cùng x            PASS -> 0
tọa độ 10^9                PASS -> 999999999
y không ảnh hưởng          PASS -> 4


In [19]:
import random

rng = random.Random(1637)
for _ in range(400):
    n = rng.randint(2, 12)
    pts = [[rng.randint(0, 30), rng.randint(0, 30)] for _ in range(n)]
    ref = widest_bruteforce(pts)
    assert widest_sort(pts) == ref, pts
    assert widest_set(pts) == ref, pts
    assert widest_bucket(pts) == ref, pts
    assert Solution().maxWidthOfVerticalArea(pts) == ref, pts
print("Brute force, sort, set, bucket và Solution khớp nhau trên 400 input")

Brute force, sort, set, bucket và Solution khớp nhau trên 400 input


In [20]:
# kiểm tra bucket trên dữ liệu lớn, tọa độ rộng
big = [[rng.randint(0, 10**9), 0] for _ in range(10**5)]
assert widest_bucket(big) == widest_sort(big)
print("Bucket khớp sort với n = 100000, đáp án =", widest_sort(big))

Bucket khớp sort với n = 100000, đáp án = 108565


## 12. Lỗi người mới hay gặp

### Lỗi 1: Sắp theo cả điểm thay vì chỉ x, rồi trừ cả list

`sorted(points)` vẫn đúng thứ tự theo x (vì list so sánh phần tử đầu trước), nhưng nếu quên `[0]` khi trừ sẽ lỗi `TypeError`.

In [21]:
pts = sorted([[3, 1], [1, 0], [5, 3]])
print("sorted(points):", pts)
try:
    print(pts[1] - pts[0])
except TypeError as e:
    print("Lỗi:", e)
print("Sửa:", pts[1][0] - pts[0][0])

sorted(points): [[1, 0], [3, 1], [5, 3]]
Lỗi: unsupported operand type(s) for -: 'list' and 'list'
Sửa: 2


`try / except` ở trên bắt lỗi để notebook vẫn chạy tiếp: nếu khối `try` gây `TypeError`, Python nhảy vào `except` và in thông báo.

### Lỗi 2: Tưởng phải dùng y, tính khoảng cách Euclid

Sai: `((x2-x1)**2 + (y2-y1)**2) ** 0.5` — đề hỏi **bề rộng dải dọc**, chỉ phụ thuộc x.

In [22]:
def wrong_distance(points):
    pts = sorted(points)
    best = 0
    for (x1, y1), (x2, y2) in zip(pts, pts[1:]):
        best = max(best, ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5)
    return best

print("sai :", wrong_distance([[8, 7], [9, 9], [7, 4], [9, 7]]))
print("đúng:", widest_sort([[8, 7], [9, 9], [7, 4], [9, 7]]))

sai : 3.1622776601683795
đúng: 1


### Lỗi 3: Lấy `max(xs) - min(xs)`

Đó là bề rộng **toàn bộ** vùng chứa điểm, không phải khe trống. Bên trong vẫn có điểm.

In [23]:
xs = [1, 1, 3, 5, 8, 9]
print("sai :", max(xs) - min(xs))
print("đúng:", max(b - a for a, b in zip(xs, xs[1:])))

sai : 8
đúng: 3


### Lỗi 4: Quên sort, trừ các phần tử kề theo thứ tự gốc

Thứ tự trong input là tùy ý. Hiệu kề khi chưa sắp có thể âm hoặc bỏ sót khe thật.

In [24]:
def wrong_no_sort(points):
    xs = [x for x, _ in points]
    return max(b - a for a, b in zip(xs, xs[1:]))

print("sai :", wrong_no_sort([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))
print("đúng:", widest_sort([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

sai : 6
đúng: 3


### Lỗi 5: Dùng `set` mà quên trường hợp còn một giá trị

`max()` trên dãy rỗng sẽ báo `ValueError`. Cách sort không bỏ trùng tránh được lỗi này vì `n >= 2` luôn cho ít nhất một cặp.

In [25]:
xs = sorted({5, 5})
try:
    print(max(b - a for a, b in zip(xs, xs[1:])))
except ValueError as e:
    print("Lỗi:", e)

Lỗi: max() iterable argument is empty


## 13. Mẹo nhận dạng pattern

- Đề hình học nhưng một chiều "vô hạn" → chiều đó **không quan trọng**, chiếu xuống chiều còn lại.
- "Khoảng trống lớn nhất / không có phần tử nào ở giữa" → **sort rồi xét phần tử liền kề**.
- "Max gap" mà cần `O(n)` → **bucket / pigeonhole**.
- Sau khi sort, nhiều bài chỉ cần nhìn `a[i]` và `a[i-1]`: Minimum Absolute Difference (1200), Maximum Gap (164), Contains Duplicate (217).

## 14. Biến thể

### Biến thể 1: Dải ngang rộng nhất (horizontal area)

Đổi vai trò x và y.

In [26]:
def widest_horizontal(points):
    ys = sorted(y for _, y in points)
    return max(b - a for a, b in zip(ys, ys[1:]))

print(widest_horizontal([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

4


### Biến thể 2: Trả về cả vị trí dải (mép trái, mép phải)

In [27]:
def widest_with_position(points):
    xs = sorted(x for x, _ in points)
    best, left = -1, None
    for a, b in zip(xs, xs[1:]):
        if b - a > best:
            best, left = b - a, a
    return best, (left, left + best)

print(widest_with_position([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

(3, (5, 8))


### Biến thể 3: Khe hẹp nhất giữa hai x **khác nhau**

Bỏ trùng trước rồi lấy `min` của hiệu kề.

In [28]:
def narrowest_distinct(points):
    xs = sorted({x for x, _ in points})
    return min(b - a for a, b in zip(xs, xs[1:]))

print(narrowest_distinct([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]))

1


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Tính tay

Với `points = [[5,0],[1,1],[12,3],[7,7],[1,9]]`: viết `xs` đã sắp, các khe, và đáp án.

### Bài 2 — Tách cột

In [29]:
def exercise_xs(points: list[list[int]]) -> list[int]:
    # TODO: trả về list x đã sắp tăng dần
    pass

# assert exercise_xs([[8, 7], [9, 9], [7, 4], [9, 7]]) == [7, 8, 9, 9]

### Bài 3 — Viết lại lời giải chính bằng vòng for chỉ số

In [30]:
def exercise_widest(points: list[list[int]]) -> int:
    # TODO 1: lấy xs và sort
    # TODO 2: duyệt i từ 1, cập nhật best = max(best, xs[i] - xs[i-1])
    pass

# assert exercise_widest([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]) == 3

### Bài 4 — Minimum Absolute Difference (LeetCode 1200 rút gọn)

Cho `arr` các số **khác nhau**, trả về hiệu nhỏ nhất giữa hai phần tử bất kỳ.

In [31]:
def exercise_min_diff(arr: list[int]) -> int:
    # TODO: sort rồi xét các cặp kề
    pass

# assert exercise_min_diff([4, 2, 1, 3]) == 1
# assert exercise_min_diff([1, 3, 6, 10, 15]) == 2

### Bài 5 — Đếm số khe đạt độ rộng lớn nhất

In [32]:
def exercise_count_widest(points: list[list[int]]) -> int:
    # TODO: tìm best, rồi đếm có bao nhiêu khe kề bằng best
    pass

# assert exercise_count_widest([[8, 7], [9, 9], [7, 4], [9, 7]]) == 2

### Bài 6 — Mini challenge: Maximum Gap `O(n)` cho list số

Không dùng `sort`. Cài đặt kỹ thuật bucket cho `nums` (số nguyên không âm), trả về khe lớn nhất giữa hai phần tử kề sau khi sắp; nếu `len(nums) < 2` trả `0`.

In [33]:
def exercise_maximum_gap(nums: list[int]) -> int:
    # TODO: lo, hi, size, các bucket min/max, duyệt bucket có dữ liệu
    pass

# assert exercise_maximum_gap([3, 6, 9, 1]) == 3
# assert exercise_maximum_gap([10]) == 0

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: bỏ y.
- Mức 2: `xs = [1, 1, 5, 7, 12]`.
- Mức 3: khe `0, 4, 2, 5` → đáp án `5`.

**Bài 2**
- Mức 1: list comprehension.
- Mức 2: `[p[0] for p in points]`.
- Mức 3: bọc bằng `sorted(...)`.

**Bài 3**
- Mức 1: bắt đầu `i` từ 1.
- Mức 2: `best = 0` trước vòng lặp.
- Mức 3: `for i in range(1, len(xs)): best = max(best, xs[i] - xs[i-1])`.

**Bài 4**
- Mức 1: hiệu nhỏ nhất luôn nằm giữa hai phần tử kề sau khi sort.
- Mức 2: dùng `zip(a, a[1:])`.
- Mức 3: `min(b - a for a, b in zip(s, s[1:]))` với `s = sorted(arr)`.

**Bài 5**
- Mức 1: hai lượt: tìm max, rồi đếm.
- Mức 2: lưu danh sách `gaps`.
- Mức 3: `gaps.count(max(gaps))`.

**Bài 6**
- Mức 1: khe lớn nhất `>=` trung bình `(hi-lo)/(m-1)`.
- Mức 2: `size = max(1, (hi - lo) // (m - 1))`, chỉ số bucket `(x - lo) // size`.
- Mức 3: lưu min/max mỗi bucket, đáp án là max của `min(bucket hiện tại) - max(bucket có dữ liệu trước đó)`.

## 17. Đáp án tham khảo

**Bài 1:** `xs = [1, 1, 5, 7, 12]`, khe `0, 4, 2, 5`, đáp án **5** (dải `7 < x < 12`).

In [34]:
def answer_xs(points):
    return sorted(p[0] for p in points)

def answer_widest(points):
    xs = sorted(p[0] for p in points)
    best = 0
    for i in range(1, len(xs)):
        best = max(best, xs[i] - xs[i - 1])
    return best

def answer_min_diff(arr):
    s = sorted(arr)
    return min(b - a for a, b in zip(s, s[1:]))

def answer_count_widest(points):
    xs = sorted(p[0] for p in points)
    gaps = [b - a for a, b in zip(xs, xs[1:])]
    return gaps.count(max(gaps))

def answer_maximum_gap(nums):
    if len(nums) < 2:
        return 0
    lo, hi, m = min(nums), max(nums), len(nums)
    size = max(1, (hi - lo) // (m - 1))
    count = (hi - lo) // size + 1
    bmin, bmax = [None] * count, [None] * count
    for x in nums:
        i = (x - lo) // size
        bmin[i] = x if bmin[i] is None else min(bmin[i], x)
        bmax[i] = x if bmax[i] is None else max(bmax[i], x)
    best, prev = 0, lo
    for i in range(count):
        if bmin[i] is not None:
            best = max(best, bmin[i] - prev)
            prev = bmax[i]
    return best

assert answer_widest([[5, 0], [1, 1], [12, 3], [7, 7], [1, 9]]) == 5   # Bài 1
assert answer_xs([[8, 7], [9, 9], [7, 4], [9, 7]]) == [7, 8, 9, 9]
assert answer_widest([[3, 1], [9, 0], [1, 0], [1, 4], [5, 3], [8, 8]]) == 3
assert answer_min_diff([4, 2, 1, 3]) == 1
assert answer_min_diff([1, 3, 6, 10, 15]) == 2
assert answer_count_widest([[8, 7], [9, 9], [7, 4], [9, 7]]) == 2
assert answer_maximum_gap([3, 6, 9, 1]) == 3
assert answer_maximum_gap([10]) == 0
rng2 = random.Random(164)
for _ in range(300):
    arr = [rng2.randint(0, 50) for _ in range(rng2.randint(2, 15))]
    s = sorted(arr)
    assert answer_maximum_gap(arr) == max(b - a for a, b in zip(s, s[1:]))
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxWidthOfVerticalArea(self, points: list[list[int]]) -> int:
        xs = sorted(x for x, _ in points)              # 1) bỏ y, sắp x
        return max(b - a for a, b in zip(xs, xs[1:]))  # 2) khe kề lớn nhất
```

Câu thần chú: **"Dải dọc vô hạn → bỏ y → sort x → max hiệu kề."**

Checklist trước khi submit:

- [ ] Chỉ dùng tọa độ x?
- [ ] Đã sort trước khi lấy hiệu kề?
- [ ] Lấy hiệu **kề** chứ không phải `max - min`?
- [ ] Nếu bỏ trùng bằng `set`, đã xử lý trường hợp chỉ còn 1 giá trị?
- [ ] Không dùng khoảng cách Euclid?
- [ ] Đã thử ví dụ có x trùng nhau (khe `0`)?